# Impala67 – Handschrift-Training in Google Colab

1. Oben **Laufzeit → Laufzeittyp ändern → T4 GPU** wählen.
2. **Laufzeit → Alle ausführen**.
3. Beim ersten Mal: Google Drive freigeben und `benchmark_test_strokes.json` hochladen (liegt lokal in `handwriting/data/`).

Alles Wichtige (Zwischenstände, MathWriting-Cache, fertiges Modell, Log) landet in Google Drive unter `Impala67-Handschrift/`.
Bricht die Sitzung ab: einfach erneut **Alle ausführen** – das Training macht beim letzten Zwischenstand weiter.
Neues Training von vorn: `Impala67-Handschrift/checkpoints/last.pt` in Drive löschen.

In [ ]:
import os, subprocess, shutil, hashlib, glob
import torch
assert torch.cuda.is_available(), 'Keine GPU! Laufzeit -> Laufzeittyp ändern -> T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Impala67-Handschrift'
os.makedirs(f'{DRIVE}/checkpoints', exist_ok=True)

In [ ]:
REPO = '/content/Impala67'
if os.path.isdir(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q --depth 1 https://github.com/prinzpuma/Impala67.git {REPO}
!pip install -q onnx onnxruntime onnxscript
HW = f'{REPO}/handwriting'
os.makedirs(f'{HW}/data/mw_download', exist_ok=True)

In [ ]:
# Eigene Testzeilen (nur zum Messen, nie zum Trainieren)
bench_drive = f'{DRIVE}/benchmark_test_strokes.json'
if not os.path.exists(bench_drive):
    from google.colab import files
    print('Bitte handwriting/data/benchmark_test_strokes.json hochladen')
    up = files.upload()
    with open(bench_drive, 'wb') as f:
        f.write(next(iter(up.values())))
shutil.copy(bench_drive, f'{HW}/data/benchmark_test_strokes.json')
print('Benchmark bereit')

In [ ]:
# Google MathWriting (CC BY 4.0). Liegt der fertige Cache in Drive, entfällt der 3-GB-Download.
caches = glob.glob(f'{DRIVE}/mathwriting-2024-*.cache.pkl')
for c in caches:
    shutil.copy(c, f'{HW}/data/mw_download/')

if not caches and not os.path.isdir(f'{HW}/data/mw_download/mathwriting-2024'):
    tgz = '/content/mathwriting-2024.tgz'
    !wget -q -c -O {tgz} https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz
    md5 = hashlib.md5()
    with open(tgz, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 24), b''):
            md5.update(chunk)
    assert md5.hexdigest() == 'f2d59c44a545347a5f67ac70fef7a13d', 'Download beschädigt – Zelle erneut ausführen'
    !tar -xzf {tgz} -C {HW}/data/mw_download
    os.remove(tgz)

# Cache bauen (einmalig einige Minuten) und in Drive sichern
!cd {HW} && python -c "from mathwriting_loader import MathWritingDataset as M; M()"
for c in glob.glob(f'{HW}/data/mw_download/*.cache.pkl'):
    if not os.path.exists(f'{DRIVE}/{os.path.basename(c)}'):
        shutil.copy(c, DRIVE)
print('MathWriting bereit')

In [ ]:
# Training. Zwischenstände gehen nach Drive (last.pt jede Epoche, best.pt bei Verbesserung).
env = dict(os.environ, HW_CHECKPOINT_DIR=f'{DRIVE}/checkpoints', PYTHONUNBUFFERED='1')
proc = subprocess.Popen(['python', 'train.py'], cwd=HW, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
with open(f'{DRIVE}/train.log', 'a') as log:
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
assert proc.wait() == 0, 'Training mit Fehler beendet – siehe Ausgabe oben'

In [ ]:
# Fertiges Modell nach Drive kopieren und herunterladen
shutil.copy(f'{HW}/model.onnx', f'{DRIVE}/model.onnx')
print('Modell in Drive gespeichert:', f'{DRIVE}/model.onnx')
from google.colab import files
files.download(f'{HW}/model.onnx')